# Football Tracking Pipeline -- Full-Match Global Re-Linking + ID-Switch Audit

Adapted from the 3,000-frame proof-of-concept to run on the full 70,326-frame match, with
fixes accumulated from real full-match diagnostic runs. Changes from the original clip
version, in order:

1. **Cell 7 loads existing raw tracks if available instead of re-tracking from scratch.**
   Points at the unbridged `online_raw_tracks_final.pkl` (604 raw tracks) deliberately -- this
   notebook's global linking is a more rigorous replacement for the bridging heuristic, not a
   second pass on top of it. Falls back to a fresh BoT-SORT pass only if no existing raw-track
   file is found.
2. **Global linking is sparse, not dense Hungarian.** Dense `linear_sum_assignment` is ~O(n^3)
   over a per-class cost matrix -- fine for tens of tracklets on a 3,000-frame clip,
   infeasible for the thousands a full match produces. Replaced with greedy matching over a
   sparse candidate graph (bisect range queries against a per-class gap bound, not an
   all-pairs scan) -- still considers every temporally-feasible pair across the whole match at
   once, just resolved greedily instead of provably-optimally.
3. **Motion scoring uses pitch-space meters (via homography), not raw pixel space**, with a
   pixel-space fallback only when homography projection is unavailable at a boundary frame.
4. **MAX_LINK_GAP is per-class, not a single global value**, with velocity extrapolation
   capped separately. Diagnosed directly from full-match data: goalkeeper/referee tracklets
   fragment mostly from very long tracking dropouts (goalkeeper median gap ~21,000 frames),
   not short occlusion, while players fragment on a much shorter contact-driven timescale.
   Uncapped velocity extrapolation over a huge gap was throwing off predictions built from
   only an 8-frame fit -- extrapolation now freezes at the last-reached position beyond
   `VELOCITY_EXTRAPOLATION_CAP_FRAMES`.
5. `raw_track_id` is now carried through tracklet construction/splitting natively (added to
   `make_tracklet`/`slice_tracklet`), instead of being recovered after the fact via bbox
   lookup.
6. **Team-assignment logic has been removed entirely (kit-color KMeans, goalkeeper
   team-centroid merge).** Goalkeeper appearance-based linking was diagnosed as unreliable
   (p50 score 0.233, p75 0.512, no clean separation between genuine and noise pairs), and the
   team-based bypass built to work around that has been dropped rather than kept. The current
   priority is zero ID switches, not hitting the exact ground-truth identity count --
   fragmentation (too many IDs) is an acceptable, recoverable outcome; a false merge (one ID
   silently spanning two real people) is not. Goalkeeper now goes through the same
   appearance/motion-based linking as player and referee, with no class-specific bypass.
   Expect more goalkeeper/referee fragmentation than before as a direct result of this.
7. **A per-ID count/QA cell** was added after ghost-track removal (detection counts, class-vote
   purity, frame span, coverage ratio) to make thin, sparse, or contested IDs easy to spot
   before or alongside auditing.



## Cell 1 -- Config

In [1]:
import os
import cv2
import pickle
import bisect
import colorsys
import numpy as np
from pathlib import Path
from collections import defaultdict, deque, Counter

# ---- full-match paths ----
DETECTION_CACHE_PATH = "/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/barca_atletico_firsthalf_detection_cache_ft3.pkl"
OSNET_WEIGHTS_PATH    = "/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/osnet_x1_0_sportsmot_best.pt"
VIDEO_PATH             = "/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/barca_atletico_46m53s.mkv"
HOMOGRAPHY_PATH         = "/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/homography_cache_barca_atletico_firsthalf.pkl"

# unbridged raw tracks -- deliberately NOT the bridged file, see notebook intro
UNBRIDGED_RAW_TRACKS_PATH = "/kaggle/input/datasets/zeinsaad/barca-atletico-online-raw-tracks/online_raw_tracks_final.pkl"

OUTPUT_TRACKING_CACHE_PATH = "/kaggle/working/barca_atletico_tracking_cache_final.pkl"
OUTPUT_VIDEO_PATH = "/kaggle/working/barca_atletico_annotated_final.mp4"

DEVICE = 0   # Kaggle GPU index, or "cpu"

CLASS_TO_ID = {"player": 0, "goalkeeper": 1, "referee": 2}
ID_TO_CLASS = {v: k for k, v in CLASS_TO_ID.items()}

PX_PER_METER = 10
PITCH_LENGTH = 105.0
PITCH_WIDTH = 68.0

# ---- tracklet splitting: force a split at every same-class contact ----
CONTACT_IOU_THRESH = 0.3
# NOTE: Stage 1's separate tracklet pipeline uses 0.1 for the same concept. Not yet
# reconciled between the two notebooks -- revisit if this notebook's splitting looks too
# permissive compared to Stage 1's tracklets on the same raw tracks.

# ---- pre-link noise filtering ----
MIN_TRACKLET_LEN = 20

# ---- global linking parameters ----
MAX_LINK_GAP_BY_CLASS = {
    "player": 500,
    "goalkeeper": 70326,
    "referee": 70326,
}
# Per-class, not a single global value. Diagnosed on real full-match data: goalkeeper
# tracklets fragment mostly from very long tracking dropouts (median gap ~21,234 frames,
# only 2.9% of all goalkeeper pairs fell within a flat 500-frame cap), not short occlusion --
# a keeper spends long stretches stationary near their own goal, often off-camera or too
# small/still for the raw tracker to hold a continuous ID through TV-style zoom/pan coverage.
# Referees patrol the whole match similarly. Players fragment on a fundamentally different,
# much shorter timescale (contact events), and actually move, so long-gap extrapolation is
# riskier for them -- left at 500.
# Goalkeeper now goes through the SAME appearance-based linking path as player/referee --
# team-assignment bypass logic has been removed (see notebook intro). Its long gap here is
# still deliberate: keepers fragment mainly from long dropouts, not short occlusion.

VELOCITY_EXTRAPOLATION_CAP_FRAMES = 250
# A velocity estimate fit over just an 8-frame window (EMBED_WINDOW) gets multiplied by
# whatever the gap is -- with per-class gaps reaching tens of thousands of frames, any tiny
# residual noise turns into a wildly wrong predicted position. Beyond this many frames,
# extrapolation freezes at the last-reached predicted position instead of continuing to
# project indefinitely.

MIN_LINK_SCORE = 0.3
EMBED_WINDOW = 8
MOTION_WEIGHT = 0.3
MOTION_NORM_M = 8.0          # meters; rough carry-over guess, validate against real distances
MOTION_NORM_PX = 300.0       # fallback only, used when homography projection is unavailable

# ---- final ghost-track filter + ratio-aware class locking ----
MIN_TRACK_LENGTH = 300
MIN_CONFIRM_FRAMES_ABS = 10
MIN_CONFIRM_RATIO = 0.02
MAX_IDS_PER_CLASS_EXPECTED = {"goalkeeper": 2, "referee": 3}   # sanity check only, not enforced

DEBUG_LINKING = False
# was True for the 3,000-frame clip -- full match will produce far too many links to
# usefully eyeball. Re-enable only when debugging a specific class/pair.

print("Checking paths...")
for p in [DETECTION_CACHE_PATH, OSNET_WEIGHTS_PATH, VIDEO_PATH, HOMOGRAPHY_PATH, UNBRIDGED_RAW_TRACKS_PATH]:
    print(p, "->", "OK" if os.path.exists(p) else "MISSING")

NEED_FRESH_TRACKING = not os.path.exists(UNBRIDGED_RAW_TRACKS_PATH)
print("")
print("Existing raw tracks found: " + str(not NEED_FRESH_TRACKING))
print("Will " + ("re-track from scratch (BoT-SORT)." if NEED_FRESH_TRACKING else "load existing raw tracks (skip re-tracking)."))


Checking paths...
/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/barca_atletico_firsthalf_detection_cache_ft3.pkl -> OK
/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/osnet_x1_0_sportsmot_best.pt -> OK
/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/barca_atletico_46m53s.mkv -> OK
/kaggle/input/datasets/zeinsaad/barca-atletico-first-half/homography_cache_barca_atletico_firsthalf.pkl -> OK
/kaggle/input/datasets/zeinsaad/barca-atletico-online-raw-tracks/online_raw_tracks_final.pkl -> OK

Existing raw tracks found: True
Will load existing raw tracks (skip re-tracking).


## Cell 2 -- Load detection cache + homography cache
Loaded unconditionally: `detection_cache` is needed later for ball detections and for kit-
color feature extraction regardless of tracking source; `homography_cache` is needed for
pitch-space motion scoring and for team-centroid computation.

In [2]:
with open(DETECTION_CACHE_PATH, "rb") as f:
    detection_cache = pickle.load(f)

with open(HOMOGRAPHY_PATH, "rb") as f:
    homography_cache = pickle.load(f)

print(f"Detection cache loaded: {len(detection_cache)} frames")
n_valid_h = sum(1 for h in homography_cache if h is not None)
print(f"Homography cache loaded: {len(homography_cache)} frames, valid: {n_valid_h}/{len(homography_cache)}")


Detection cache loaded: 70326 frames
Homography cache loaded: 70327 frames, valid: 70326/70327


## Cell 3 -- Load fine-tuned OSNet (only if re-tracking from scratch)
Skipped entirely when existing raw tracks are found -- embeddings are already computed and
stored per detection in that case.

In [3]:
if NEED_FRESH_TRACKING:
    !pip install -q torchreid

    import torch, torch.nn as nn
    import torchreid
    import torchvision.transforms as T
    from PIL import Image

    ckpt = torch.load(OSNET_WEIGHTS_PATH, map_location="cpu")
    state_dict = ckpt.get("state_dict", ckpt.get("model", ckpt)) if isinstance(ckpt, dict) else ckpt

    osnet = torchreid.models.build_model(name="osnet_x1_0", num_classes=302, pretrained=False)
    osnet.load_state_dict(state_dict, strict=True)
    osnet.classifier = nn.Identity()
    osnet.eval()
    osnet.to("cuda" if DEVICE != "cpu" else "cpu")

    osnet_transform = T.Compose([
        T.Resize((256, 128)), T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    print("OSNet loaded for fresh tracking pass. Embedding dim: 512")
else:
    print("Existing raw tracks found -- skipping OSNet load (not needed).")


Existing raw tracks found -- skipping OSNet load (not needed).


## Cell 4 -- Install boxmot + initialize tracker (only if re-tracking from scratch)
**Never re-run `pip install boxmot` without `--no-deps`** -- it will downgrade torch/numpy.

In [4]:
if NEED_FRESH_TRACKING:
    !pip install -q boxmot==10.0.84 --no-deps
    !pip install -q loguru ftfy regex lap filterpy --no-deps

    from boxmot import BoTSORT

    device_str = "cpu" if DEVICE == "cpu" else str(DEVICE)

    tracker = BoTSORT(
        model_weights=Path(OSNET_WEIGHTS_PATH),
        device=device_str,
        fp16=False,
        track_high_thresh=0.5,
        track_low_thresh=0.1,
        new_track_thresh=0.6,
        track_buffer=100,
        match_thresh=0.8,
        proximity_thresh=0.5,
        appearance_thresh=0.25,
        cmc_method="sof",
        frame_rate=25,
    )
    print(f"BoT-SORT initialized on device={device_str}")
else:
    print("Existing raw tracks found -- skipping BoT-SORT init.")


Existing raw tracks found -- skipping BoT-SORT init.


## Cell 5 -- Core functions: tracklet building, contact splitting, pitch-space
features, sparse global linking, kit-color feature extraction
Defined once, upfront, and validated against synthetic data in Cell 6 before ever touching
real data. `raw_track_id` is now carried through `make_tracklet`/`slice_tracklet` natively.

In [5]:
def iou(box1, box2):
    xa1, ya1, xa2, ya2 = box1; xb1, yb1, xb2, yb2 = box2
    ix1, iy1 = max(xa1, xb1), max(ya1, yb1)
    ix2, iy2 = min(xa2, xb2), min(ya2, yb2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    union = (xa2 - xa1) * (ya2 - ya1) + (xb2 - xb1) * (yb2 - yb1) - inter
    return inter / union if union > 0 else 0.0


def bbox_center(bbox):
    x1, y1, x2, y2 = bbox
    return np.array([(x1 + x2) / 2, (y1 + y2) / 2])


def foot_anchor(bbox):
    x1, y1, x2, y2 = bbox
    return (x1 + x2) / 2, y2


def project_to_pitch(H, x, y):
    """Project an image-space point to pitch-space meters via homography."""
    pt = cv2.perspectiveTransform(np.array([[[x, y]]], dtype=np.float32), np.array(H)).reshape(2)
    return np.array([pt[0] / PX_PER_METER, pt[1] / PX_PER_METER])


def pitch_position(frame_idx, bbox, hcache):
    """Returns pitch-space (meters) position, or None if homography is unavailable for this frame."""
    if frame_idx == 0 or frame_idx >= len(hcache) or hcache[frame_idx] is None:
        return None
    fx, fy = foot_anchor(bbox)
    return project_to_pitch(hcache[frame_idx], fx, fy)


def make_tracklet(entries, raw_track_id):
    """entries: list of (frame_idx, bbox, conf, cls, embedding), sorted by frame_idx."""
    frames = [e[0] for e in entries]
    bbox_by_frame = {e[0]: e[1] for e in entries}
    conf_by_frame = {e[0]: e[2] for e in entries}
    class_by_frame = {e[0]: e[3] for e in entries}
    embedding_by_frame = {e[0]: e[4] for e in entries}
    classes = [e[3] for e in entries]
    majority_class = max(set(classes), key=classes.count)
    return {
        "frames": frames, "bbox_by_frame": bbox_by_frame, "conf_by_frame": conf_by_frame,
        "class_by_frame": class_by_frame, "embedding_by_frame": embedding_by_frame,
        "class": majority_class, "raw_track_id": raw_track_id,
    }


def build_initial_tracklets(raw_tracks_by_frame):
    """Groups raw per-frame tracker output by raw_track_id. A raw ID's lifespan is already
    one continuous stretch with no ambiguity according to the tracker -- exactly what a
    tracklet should be before contact-based splitting. The frame-gap segmentation below is a
    no-op safety net for genuinely continuous raw tracker output (BotSort assigns a NEW raw ID
    after any real break, so an unbridged raw track should never contain an internal gap) --
    it is NOT the same bug as splitting a BRIDGED track on gaps, which would wrongly undo a
    deliberate cross-occlusion merge. This notebook loads the UNBRIDGED file, so that concern
    does not apply here."""
    raw_entries_by_id = defaultdict(list)
    for frame_idx, dets in raw_tracks_by_frame.items():
        for d in dets:
            raw_entries_by_id[d["raw_track_id"]].append(
                (frame_idx, d["bbox"], d["conf"], d["class"], d["embedding"])
            )

    tracklets = []
    for raw_id, entries in raw_entries_by_id.items():
        entries.sort(key=lambda e: e[0])
        segments, current = [], [entries[0]]
        for e in entries[1:]:
            if e[0] == current[-1][0] + 1:
                current.append(e)
            else:
                segments.append(current)
                current = [e]
        segments.append(current)
        for seg in segments:
            tracklets.append(make_tracklet(seg, raw_id))
    return tracklets


def find_contact_split_points(tracklets, iou_thresh, merge_gap=1):
    """Returns {tracklet_index: set of cut-boundary frames}. A cut boundary at frame f means
    end a chunk after frame f -- so the tracklet is split right before and right after each
    contact event, isolating the ambiguous contact frames into their own short segment instead
    of leaving them attached to a clean pre/post segment.

    Contact frames are grouped into contiguous runs (allowing gaps of up to merge_gap frames to
    still count as one run) before computing boundaries -- otherwise an extended contact (many
    consecutive overlapping frames) would get a cut point at EVERY overlapping frame, shredding
    the tracklet into single-frame fragments that then get dropped entirely by MIN_TRACKLET_LEN,
    silently deleting real player data for that whole stretch.
    """
    overlap_frames_by_pair = defaultdict(set)
    frame_to_tracklets = defaultdict(list)
    for idx, tl in enumerate(tracklets):
        for f in tl["frames"]:
            frame_to_tracklets[f].append(idx)

    for f, idxs in frame_to_tracklets.items():
        for i in range(len(idxs)):
            for j in range(i + 1, len(idxs)):
                a, b = tracklets[idxs[i]], tracklets[idxs[j]]
                if a["class"] != b["class"]:
                    continue
                if iou(a["bbox_by_frame"][f], b["bbox_by_frame"][f]) >= iou_thresh:
                    pair = tuple(sorted((idxs[i], idxs[j])))
                    overlap_frames_by_pair[pair].add(f)

    split_points = defaultdict(set)
    for (i, j), frames in overlap_frames_by_pair.items():
        frames = sorted(frames)
        runs, current = [], [frames[0]]
        for f in frames[1:]:
            if f - current[-1] <= merge_gap:
                current.append(f)
            else:
                runs.append(current)
                current = [f]
        runs.append(current)
        for run in runs:
            split_points[i].add(run[0] - 1)
            split_points[i].add(run[-1])
            split_points[j].add(run[0] - 1)
            split_points[j].add(run[-1])
    return split_points


def slice_tracklet(tl, frames):
    return {
        "frames": frames,
        "bbox_by_frame": {f: tl["bbox_by_frame"][f] for f in frames},
        "conf_by_frame": {f: tl["conf_by_frame"][f] for f in frames},
        "class_by_frame": {f: tl["class_by_frame"][f] for f in frames},
        "embedding_by_frame": {f: tl["embedding_by_frame"][f] for f in frames},
        "class": tl["class"],
        "raw_track_id": tl["raw_track_id"],
    }


def apply_splits(tracklets, split_points):
    new_tracklets = []
    for idx, tl in enumerate(tracklets):
        cuts = set(split_points.get(idx, []))
        if not cuts:
            new_tracklets.append(tl)
            continue
        chunks, current = [], []
        for f in tl["frames"]:
            current.append(f)
            if f in cuts:
                chunks.append(current)
                current = []
        if current:
            chunks.append(current)
        for chunk in chunks:
            if chunk:
                new_tracklets.append(slice_tracklet(tl, chunk))
    return new_tracklets


def fit_velocity(frames, bbox_by_frame):
    """Pixel-space fallback velocity fit -- only used when pitch-space projection is
    unavailable for enough of the window."""
    if len(frames) < 2:
        return None
    farr = np.array(frames, dtype=np.float64)
    centers = np.array([bbox_center(bbox_by_frame[f]) for f in frames])
    A = np.vstack([farr, np.ones_like(farr)]).T
    vx, cx = np.linalg.lstsq(A, centers[:, 0], rcond=None)[0]
    vy, cy = np.linalg.lstsq(A, centers[:, 1], rcond=None)[0]
    return np.array([vx, vy])


def fit_velocity_pitch(frames, bbox_by_frame, hcache):
    """Pitch-space (meters) velocity fit via homography. Returns None if fewer than 2 frames
    in the window have valid homography projection."""
    pts = []
    for f in frames:
        p = pitch_position(f, bbox_by_frame[f], hcache)
        if p is not None:
            pts.append((f, p))
    if len(pts) < 2:
        return None
    farr = np.array([p[0] for p in pts], dtype=np.float64)
    parr = np.array([p[1] for p in pts])
    A = np.vstack([farr, np.ones_like(farr)]).T
    vx, cx = np.linalg.lstsq(A, parr[:, 0], rcond=None)[0]
    vy, cy = np.linalg.lstsq(A, parr[:, 1], rcond=None)[0]
    return np.array([vx, vy])


def compute_tracklet_features(tl, window, hcache):
    frames = tl["frames"]
    head_frames = frames[:window]
    tail_frames = frames[-window:]

    head_embs = [tl["embedding_by_frame"][f] for f in head_frames if tl["embedding_by_frame"].get(f) is not None]
    tail_embs = [tl["embedding_by_frame"][f] for f in tail_frames if tl["embedding_by_frame"].get(f) is not None]
    tl["head_emb"] = np.mean(head_embs, axis=0) if head_embs else None
    tl["tail_emb"] = np.mean(tail_embs, axis=0) if tail_embs else None

    # pixel-space (fallback)
    tl["head_vel"] = fit_velocity(head_frames, tl["bbox_by_frame"])
    tl["tail_vel"] = fit_velocity(tail_frames, tl["bbox_by_frame"])
    tl["head_pos"] = bbox_center(tl["bbox_by_frame"][frames[0]])
    tl["tail_pos"] = bbox_center(tl["bbox_by_frame"][frames[-1]])

    # pitch-space (preferred) -- None if homography unavailable for this tracklet's boundary
    tl["head_vel_m"] = fit_velocity_pitch(head_frames, tl["bbox_by_frame"], hcache)
    tl["tail_vel_m"] = fit_velocity_pitch(tail_frames, tl["bbox_by_frame"], hcache)
    tl["head_pos_m"] = pitch_position(frames[0], tl["bbox_by_frame"][frames[0]], hcache)
    tl["tail_pos_m"] = pitch_position(frames[-1], tl["bbox_by_frame"][frames[-1]], hcache)


def link_score(ti, tj, motion_weight, motion_norm_m, motion_norm_px, max_gap,
               velocity_extrapolation_cap_frames=250):
    gap = tj["frames"][0] - ti["frames"][-1]
    if gap <= 0 or gap > max_gap:
        return None

    if ti["tail_emb"] is None or tj["head_emb"] is None:
        appearance_sim = 0.0
    else:
        appearance_sim = float(np.dot(ti["tail_emb"], tj["head_emb"]))

    # Cap how far a velocity estimate is allowed to extrapolate. A velocity fit over an
    # 8-frame window is noisy; multiplying that noise by a gap of tens of thousands of frames
    # can throw the prediction wildly off. Beyond this cap, extrapolation freezes at the
    # last-reached predicted position instead of continuing indefinitely.
    extrap_gap = min(gap, velocity_extrapolation_cap_frames)

    # Prefer pitch-space (metric, perspective-corrected) motion scoring. Pixel-space is a
    # fallback only for boundary frames where homography projection failed -- pixel distance
    # is not comparable in meaning across different gap sizes the way pitch meters are, so
    # this fallback is strictly weaker evidence and only used when pitch space is unavailable.
    if ti["tail_pos_m"] is not None and tj["head_pos_m"] is not None:
        vel = ti["tail_vel_m"] if ti["tail_vel_m"] is not None else np.zeros(2)
        predicted = ti["tail_pos_m"] + vel * extrap_gap
        dist = float(np.linalg.norm(predicted - tj["head_pos_m"]))
        motion_penalty = min(dist / motion_norm_m, 1.0)
    else:
        vel = ti["tail_vel"] if ti["tail_vel"] is not None else np.zeros(2)
        predicted = ti["tail_pos"] + vel * extrap_gap
        dist = float(np.linalg.norm(predicted - tj["head_pos"]))
        motion_penalty = min(dist / motion_norm_px, 1.0)

    return appearance_sim - motion_weight * motion_penalty


def link_tracklets_globally_sparse(tracklets, max_gap_by_class, min_link_score, motion_weight,
                                    motion_norm_m, motion_norm_px,
                                    velocity_extrapolation_cap_frames=250,
                                    default_max_gap=500, classes_to_skip=None, debug=False):
    """
    Global (whole-match, not sliding-window) candidate consideration, resolved via greedy
    matching over a SPARSE candidate graph -- replaces dense per-class Hungarian assignment
    (scipy.optimize.linear_sum_assignment), which is ~O(n^3) and only tractable at
    3,000-frame-clip scale. At full-match scale (thousands of tracklets per class), dense
    Hungarian would never finish.

    max_gap_by_class: dict mapping class name -> max linkable gap in frames. Different classes
    fragment on very different timescales (goalkeeper/referee: long tracking dropouts, tens of
    thousands of frames; player: short contact-driven splits). A class missing from the dict
    falls back to default_max_gap.

    classes_to_skip: classes to exclude from appearance-based linking entirely (e.g.
    "goalkeeper", where identity resolution is instead handled via team-based merge -- see
    Cell 13 -- because appearance alone does not cleanly separate genuine same-person pairs
    for that class on real full-match data).

    Candidate generation uses bisect range queries against each class's max gap (not an
    all-pairs scan), so it stays near-linear in the number of tracklets plus the number of
    actually temporally-feasible pairs. Resolution is greedy (highest score first, each
    tracklet claimed at most once as a source and at most once as a target) rather than
    provably globally-optimal -- a deliberate tractability tradeoff, not an oversight.
    """
    classes_to_skip = classes_to_skip or set()
    links = {}
    by_class = defaultdict(list)
    for idx, tl in enumerate(tracklets):
        by_class[tl["class"]].append(idx)

    for cls, idxs in by_class.items():
        if cls in classes_to_skip:
            print(f"class={cls}: skipped (appearance-based linking disabled for this class)")
            continue

        max_gap = max_gap_by_class.get(cls, default_max_gap)
        idxs_by_start = sorted(idxs, key=lambda i: tracklets[i]["frames"][0])
        start_frames = [tracklets[i]["frames"][0] for i in idxs_by_start]

        candidates = []
        for i in idxs:
            end_frame = tracklets[i]["frames"][-1]
            lo = bisect.bisect_right(start_frames, end_frame)
            hi = bisect.bisect_right(start_frames, end_frame + max_gap)
            for k in range(lo, hi):
                j = idxs_by_start[k]
                if j == i:
                    continue
                score = link_score(tracklets[i], tracklets[j], motion_weight, motion_norm_m,
                                    motion_norm_px, max_gap, velocity_extrapolation_cap_frames)
                if score is not None and score >= min_link_score:
                    candidates.append((score, i, j))

        candidates.sort(key=lambda c: -c[0])
        used_as_source, used_as_target = set(), set()
        n_class_links = 0
        for score, i, j in candidates:
            if i in used_as_source or j in used_as_target:
                continue
            links[i] = j
            used_as_source.add(i)
            used_as_target.add(j)
            n_class_links += 1
            if debug:
                print(f"  [link] class={cls}: tracklet {i} (ends frame {tracklets[i]['frames'][-1]}) "
                      f"-> tracklet {j} (starts frame {tracklets[j]['frames'][0]}), score={score:.3f}")
        print(f"class={cls}: {len(idxs)} tracklets, {len(candidates)} candidate pairs (max_gap={max_gap}), {n_class_links} links accepted")

    return links


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            self.parent[rx] = ry


print("Core functions ready.")


Core functions ready.


## Cell 6 -- Unit test: global linking reconnects a fragmented player and rejects a decoy
Synthetic scenario, independent of the video/detection cache: one real player is fragmented
into 3 tracklets by two separate contact events; a second, unrelated player's single tracklet
exists in the same time range. Confirms the linker chains the 3 real fragments into one
identity and does NOT accidentally link the decoy into that chain. Uses an identity
homography (pitch meters == pixel position / PX_PER_METER) so the pitch-space path is
exercised too, not just the pixel-space fallback. Run this before spending GPU time on the
full video.

In [6]:
def _run_global_linking_unit_test():
    print("Running global linking unit test...")

    rng = np.random.default_rng(7)
    emb_player = rng.normal(size=512); emb_player /= np.linalg.norm(emb_player)
    emb_decoy  = rng.normal(size=512); emb_decoy  /= np.linalg.norm(emb_decoy)

    identity_h = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)
    test_hcache = [identity_h] * 200

    def make(frames, x_start, vx, emb, cls="player"):
        bbox_by_frame, conf_by_frame, class_by_frame, embedding_by_frame = {}, {}, {}, {}
        for k, f in enumerate(frames):
            x = x_start + vx * k
            bbox_by_frame[f] = [x, 100, x + 40, 180]
            conf_by_frame[f] = 0.9
            class_by_frame[f] = cls
            embedding_by_frame[f] = emb
        return {
            "frames": frames, "bbox_by_frame": bbox_by_frame, "conf_by_frame": conf_by_frame,
            "class_by_frame": class_by_frame, "embedding_by_frame": embedding_by_frame, "class": cls,
            "raw_track_id": "synthetic",
        }

    frag1 = make(list(range(0, 20)), x_start=100, vx=2, emb=emb_player)
    frag2 = make(list(range(40, 60)), x_start=100 + 2*19 + 2*20, vx=2, emb=emb_player)
    frag3 = make(list(range(90, 110)), x_start=100 + 2*19 + 2*20 + 2*19 + 2*30, vx=2, emb=emb_player)
    decoy = make(list(range(45, 65)), x_start=500, vx=-1, emb=emb_decoy)

    test_tracklets = [frag1, frag2, frag3, decoy]
    for tl in test_tracklets:
        compute_tracklet_features(tl, window=8, hcache=test_hcache)

    # motion_norm_m=30.0 matches the original test's motion_norm_px=300 scaled by PX_PER_METER=10
    links = link_tracklets_globally_sparse(test_tracklets, max_gap_by_class={"player": 50},
                                            min_link_score=0.3, motion_weight=0.3,
                                            motion_norm_m=30.0, motion_norm_px=300.0,
                                            velocity_extrapolation_cap_frames=50, debug=False)

    test_uf = UnionFind(len(test_tracklets))
    for i, j in links.items():
        test_uf.union(i, j)
    groups = defaultdict(list)
    for idx in range(len(test_tracklets)):
        groups[test_uf.find(idx)].append(idx)

    print(f"  links: {links}")
    print(f"  groups: {dict(groups)}")

    group_sizes = sorted(len(v) for v in groups.values())
    assert group_sizes == [1, 3], f"FAIL: expected groups of size [1,3], got {group_sizes}"
    real_group = [v for v in groups.values() if len(v) == 3][0]
    assert set(real_group) == {0, 1, 2}, f"FAIL: wrong tracklets grouped: {real_group}"

    print("PASS: 3 fragments of the real player correctly chained; decoy correctly excluded.")
    print("")


_run_global_linking_unit_test()


Running global linking unit test...
class=player: 4 tracklets, 2 candidate pairs (max_gap=50), 2 links accepted
  links: {1: 2, 0: 1}
  groups: {2: [0, 1, 2], 3: [3]}
PASS: 3 fragments of the real player correctly chained; decoy correctly excluded.



## Cell 7 -- Load raw tracks if available, else run a fresh causal tracking pass
No gallery, no swap corrector, no online ID resolution -- just the raw per-frame track ID,
bbox, class, and embedding. Everything from here on operates on `raw_tracks_by_frame` using
the functions defined in Cell 5, regardless of which branch produced it.

In [7]:
if not NEED_FRESH_TRACKING:
    print(f"Loading existing raw tracks: {UNBRIDGED_RAW_TRACKS_PATH}")
    with open(UNBRIDGED_RAW_TRACKS_PATH, "rb") as f:
        raw_data = pickle.load(f)
    raw_tracks_by_frame = raw_data["raw_tracks_by_frame"]
    n_frames = raw_data["n_frames"]
    frame_idx = n_frames  # so downstream cells referencing frame_idx still work

    unique_ids = set()
    for dets in raw_tracks_by_frame.values():
        for d in dets:
            unique_ids.add(d["raw_track_id"])
    print(f"Loaded {n_frames} frames, {len(unique_ids)} unique raw track IDs.")
    assert len(unique_ids) == 604, f"Expected 604 unbridged raw tracks, got {len(unique_ids)} -- check path/mount freshness"
    print("Sanity check passed: unbridged raw tracks loaded correctly.")

else:
    print("No existing raw tracks found -- running fresh BoT-SORT pass over full video.")
    raw_tracks_by_frame = {}

    cap = cv2.VideoCapture(VIDEO_PATH)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_dets = detection_cache.get(frame_idx, [])
        person_dets = [d for d in frame_dets if d["class"] in CLASS_TO_ID]

        dets_array = (np.array([[*d["bbox"], d["conf"], CLASS_TO_ID[d["class"]]] for d in person_dets], dtype=np.float64)
                      if person_dets else np.empty((0, 6), dtype=np.float64))

        tracked = tracker.update(dets_array, frame)

        embedding_lookup = {}
        for strack in tracker.active_tracks:
            if strack.curr_feat is not None:
                emb = strack.curr_feat
                norm = np.linalg.norm(emb)
                embedding_lookup[strack.id] = emb / norm if norm > 0 else emb

        frame_entries = []
        for x1, y1, x2, y2, raw_tid, conf, cls_id, det_ind in tracked:
            raw_tid = int(raw_tid)
            bbox = [float(x1), float(y1), float(x2), float(y2)]
            cls = person_dets[int(det_ind)]["class"]   # raw per-frame class, not boxmot's smoothed label
            embedding = embedding_lookup.get(raw_tid)
            frame_entries.append({"raw_track_id": raw_tid, "bbox": bbox, "conf": float(conf),
                                   "class": cls, "embedding": embedding})

        raw_tracks_by_frame[frame_idx] = frame_entries

        if frame_idx % 200 == 0:
            print(f"frame {frame_idx}/{total_frames} | active raw tracks: {len(frame_entries)}")

        frame_idx += 1

    cap.release()
    n_frames = frame_idx
    print(f"Done. Processed {frame_idx} frames of raw tracking.")


Loading existing raw tracks: /kaggle/input/datasets/zeinsaad/barca-atletico-online-raw-tracks/online_raw_tracks_final.pkl
Loaded 70326 frames, 604 unique raw track IDs.
Sanity check passed: unbridged raw tracks loaded correctly.


## Cell 8 -- Build initial tracklets, split at contact points

In [8]:
initial_tracklets = build_initial_tracklets(raw_tracks_by_frame)
print(f"Initial tracklets (from raw track IDs): {len(initial_tracklets)}")

split_points = find_contact_split_points(initial_tracklets, CONTACT_IOU_THRESH)
split_tracklets = apply_splits(initial_tracklets, split_points)
print(f"Tracklets after contact-based splitting: {len(split_tracklets)} "
      f"(from {len(initial_tracklets)}, {sum(len(v) for v in split_points.values())} split points applied)")

pre_filter_count = len(split_tracklets)
split_tracklets = [tl for tl in split_tracklets if len(tl["frames"]) >= MIN_TRACKLET_LEN]
print(f"Dropped {pre_filter_count - len(split_tracklets)} tracklets shorter than "
      f"MIN_TRACKLET_LEN={MIN_TRACKLET_LEN} frames (too short for reliable linking)")
print(f"Tracklets going into global linking: {len(split_tracklets)}")

by_class = defaultdict(int)
for tl in split_tracklets:
    by_class[tl["class"]] += 1
print(f"By class: {dict(by_class)}")

missing_raw_id = sum(1 for tl in split_tracklets if tl.get("raw_track_id") is None)
print(f"Tracklets missing raw_track_id: {missing_raw_id}")

player_split_idxs = [i for i, tl in enumerate(split_tracklets) if tl["class"] == "player"]
gk_split_idxs = [i for i, tl in enumerate(split_tracklets) if tl["class"] == "goalkeeper"]
ref_split_idxs = [i for i, tl in enumerate(split_tracklets) if tl["class"] == "referee"]
print(f"player: {len(player_split_idxs)}, goalkeeper: {len(gk_split_idxs)}, referee: {len(ref_split_idxs)}")


Initial tracklets (from raw track IDs): 2463
Tracklets after contact-based splitting: 11154 (from 2463, 10886 split points applied)
Dropped 8050 tracklets shorter than MIN_TRACKLET_LEN=20 frames (too short for reliable linking)
Tracklets going into global linking: 3104
By class: {'referee': 198, 'player': 2805, 'goalkeeper': 101}
Tracklets missing raw_track_id: 0
player: 2805, goalkeeper: 101, referee: 198


## Cell 9 -- Compute head/tail features, then run sparse global linking (all classes)
Goalkeeper is no longer excluded -- it goes through the same appearance/motion-based linking as player and referee. Team-assignment-based bypass logic has been removed; expect goalkeeper to stay more fragmented as a result -- that is accepted, since the priority is zero ID switches over hitting the exact ground-truth count. Use the companion ID-switch audit notebook to verify no merge (for any class) fuses two different real people.

In [9]:
for tl in split_tracklets:
    compute_tracklet_features(tl, EMBED_WINDOW, homography_cache)

n_missing_pitch = sum(1 for tl in split_tracklets if tl["head_pos_m"] is None or tl["tail_pos_m"] is None)
print(f"Computed head/tail features for {len(split_tracklets)} tracklets.")
print(f"Tracklets missing pitch-space position at a boundary (pixel fallback used for motion score): {n_missing_pitch}")

accepted_links = link_tracklets_globally_sparse(
    split_tracklets, MAX_LINK_GAP_BY_CLASS, MIN_LINK_SCORE, MOTION_WEIGHT, MOTION_NORM_M, MOTION_NORM_PX,
    velocity_extrapolation_cap_frames=VELOCITY_EXTRAPOLATION_CAP_FRAMES,
    debug=DEBUG_LINKING
)  # no classes_to_skip -- goalkeeper now linked the same way as player/referee
print(f"\nTotal accepted links: {len(accepted_links)}")


Computed head/tail features for 3104 tracklets.
Tracklets missing pitch-space position at a boundary (pixel fallback used for motion score): 23
class=referee: 198 tracklets, 13816 candidate pairs (max_gap=70326), 174 links accepted
class=player: 2805 tracklets, 47049 candidate pairs (max_gap=500), 2723 links accepted
class=goalkeeper: 101 tracklets, 2343 candidate pairs (max_gap=70326), 82 links accepted

Total accepted links: 2979


## Cell 10 -- Union links into final global identities
Pure appearance/motion-based union -- no team-based goalkeeper merge. All three classes are resolved identically now. Higher fragmentation for goalkeeper/referee is expected and acceptable; run the companion audit notebook against the saved cache to confirm no merge fused two different people.

In [10]:
uf = UnionFind(len(split_tracklets))
for i, j in accepted_links.items():
    uf.union(i, j)

chain_members = defaultdict(list)
for idx in range(len(split_tracklets)):
    chain_members[uf.find(idx)].append(idx)

canonical_id_of_tracklet = {}
canonical_class = {}
next_id = 1
for root, members in chain_members.items():
    for m in members:
        canonical_id_of_tracklet[m] = next_id
    canonical_class[next_id] = split_tracklets[members[0]]["class"]
    next_id += 1

chain_lengths = [len(members) for members in chain_members.values()]
print(f"Final global identities: {len(chain_members)}")
print(f"Chain length distribution: 1 tracklet={sum(1 for c in chain_lengths if c==1)}, "
      f"2={sum(1 for c in chain_lengths if c==2)}, 3+={sum(1 for c in chain_lengths if c>=3)}")

by_class_counts = defaultdict(int)
for cid, cls in canonical_class.items():
    by_class_counts[cls] += 1
print("Identity counts by class:", dict(by_class_counts))
for cls, expected in MAX_IDS_PER_CLASS_EXPECTED.items():
    actual = by_class_counts.get(cls, 0)
    if actual > expected:
        print(f"  [check] {cls}: {actual} identities, expected <= {expected} "
              f"(fragmentation above ground truth is OK -- run the ID-switch audit notebook "
              f"before treating this as a problem)")


Final global identities: 125
Chain length distribution: 1 tracklet=11, 2=9, 3+=105
Identity counts by class: {'referee': 24, 'player': 82, 'goalkeeper': 19}
  [check] goalkeeper: 19 identities, expected <= 2 (fragmentation above ground truth is OK -- run the ID-switch audit notebook before treating this as a problem)
  [check] referee: 24 identities, expected <= 3 (fragmentation above ground truth is OK -- run the ID-switch audit notebook before treating this as a problem)


## Cell 11 -- Rebuild the final per-frame tracking cache with global IDs

In [11]:
final_tracking_cache = defaultdict(lambda: {"ball": None, "tracks": []})

for idx, tl in enumerate(split_tracklets):
    cid = canonical_id_of_tracklet[idx]
    for f in tl["frames"]:
        final_tracking_cache[f]["tracks"].append({
            "track_id": cid,
            "bbox": tl["bbox_by_frame"][f],
            "conf": tl["conf_by_frame"][f],
            "class": tl["class_by_frame"][f],
        })

for f in range(n_frames):
    ball_det = next((d for d in detection_cache.get(f, []) if d["class"] == "ball"), None)
    final_tracking_cache.setdefault(f, {"ball": ball_det, "tracks": []})
    final_tracking_cache[f]["ball"] = ball_det

final_tracking_cache = dict(final_tracking_cache)
print(f"Rebuilt tracking cache: {len(final_tracking_cache)} frames.")


Rebuilt tracking cache: 70326 frames.


## Cell 12 -- Ghost-track removal + ratio-aware class locking

In [12]:
id_frame_counts = defaultdict(int)
for data in final_tracking_cache.values():
    for t in data["tracks"]:
        id_frame_counts[t["track_id"]] += 1

ghost_ids = {tid for tid, count in id_frame_counts.items() if count < MIN_TRACK_LENGTH}
print(f"Ghost tracks dropped (< {MIN_TRACK_LENGTH} frames): {len(ghost_ids)}")

for data in final_tracking_cache.values():
    data["tracks"] = [t for t in data["tracks"] if t["track_id"] not in ghost_ids]

class_votes = defaultdict(lambda: defaultdict(int))
for data in final_tracking_cache.values():
    for t in data["tracks"]:
        class_votes[t["track_id"]][t["class"]] += 1

locked_class_by_id = {}
for tid, votes in class_votes.items():
    total = sum(votes.values())
    ref_votes = votes.get("referee", 0)
    gk_votes = votes.get("goalkeeper", 0)
    ref_confirmed = ref_votes >= MIN_CONFIRM_FRAMES_ABS and (ref_votes / total) >= MIN_CONFIRM_RATIO
    gk_confirmed = gk_votes >= MIN_CONFIRM_FRAMES_ABS and (gk_votes / total) >= MIN_CONFIRM_RATIO
    if ref_confirmed:
        locked_class_by_id[tid] = "referee"
    elif gk_confirmed:
        locked_class_by_id[tid] = "goalkeeper"
    else:
        locked_class_by_id[tid] = max(votes, key=votes.get)

final_counts = defaultdict(int)
for cls in locked_class_by_id.values():
    final_counts[cls] += 1
print(f"\nFinal identity counts: {dict(final_counts)}")


Ghost tracks dropped (< 300 frames): 13

Final identity counts: {'referee': 24, 'player': 68, 'goalkeeper': 20}


## Cell 12b -- Split confirmed ID-switches (track 2, track 41)

Found via the Cell 13b id-count screen (fresh ft3 IoU-matched class vs each referee-locked track's stored class), then pinpointed by binary search on the fresh per-frame class:

- **track 2**: referee -> goalkeeper at frame 30444 (real contamination: 3169/33572 frames, 9.4%, not noise)
- **track 41**: referee -> player at frame 34333 (real contamination: 7522/15847 frames, 47.5%)

Both are single, clean switch points in this notebook's own linking output (confirmed via coarse full-range scan before pinpointing -- no multi-switch chaining here, unlike an earlier notebook version's more fragmented linking). Each track is split at its exact switch frame: everything before stays under the original id with its original (correct) referee label; everything from the switch frame onward moves to a new synthetic id with the corrected class. The new ids still need a team assignment afterward for the ones reclassified as player/goalkeeper -- they were never in the team assigner's classification set, since they didn't exist as distinct ids when it ran.

In [13]:
# ---- Cell 12b (NEW) -- Split track 2 and track 41 at their confirmed switch
# frames. MUST run before Cell 13/13b/14 so all three reflect the corrected state.

SPLITS = [
    # (track_id, switch_frame, new_track_id, new_class)
    (2,  30444, 9002, "goalkeeper"),   # referee -> goalkeeper
    (41, 34333, 9041, "player"),        # referee -> player
]

for old_tid, switch_frame, new_tid, new_class in SPLITS:
    assert new_tid not in locked_class_by_id, f"{new_tid} already in use!"

    frames_with_old = sorted(
        f for f in final_tracking_cache
        if any(t["track_id"] == old_tid for t in final_tracking_cache[f]["tracks"])
    )

    n_kept, n_reassigned = 0, 0
    for f in frames_with_old:
        if f < switch_frame:
            n_kept += 1
            continue
        for t in final_tracking_cache[f]["tracks"]:
            if t["track_id"] != old_tid:
                continue
            t["track_id"] = new_tid
            t["class"] = new_class
            n_reassigned += 1

    locked_class_by_id[new_tid] = new_class

    print(f"track {old_tid} -> split at frame {switch_frame}: "
          f"kept {n_kept} frames as {old_tid} ({locked_class_by_id.get(old_tid)}), "
          f"reassigned {n_reassigned} frames to {new_tid} ({new_class})")

final_counts = defaultdict(int)
for cls in locked_class_by_id.values():
    final_counts[cls] += 1
print(f"\nFinal identity counts after splits: {dict(final_counts)}")

track 2 -> split at frame 30444: kept 30400 frames as 2 (referee), reassigned 3172 frames to 9002 (goalkeeper)
track 41 -> split at frame 34333: kept 8327 frames as 41 (referee), reassigned 7520 frames to 9041 (player)

Final identity counts after splits: {'referee': 24, 'player': 69, 'goalkeeper': 21}


## Cell 13 -- Per-ID summary: detection counts, class votes, span
One row per final identity, for QA before/alongside the ID-switch audit notebook -- how much data each ID actually has, whether its class label was clean or contested, and how spread out across the match it is (a huge frame span with a low detection count is a mild signal of a long-gap merge worth checking first in the audit).

In [14]:
id_summary = []
for tid, count in id_frame_counts.items():
    if tid in ghost_ids:
        continue
    frames_present = sorted(
        f for f, data in final_tracking_cache.items()
        for t in data["tracks"] if t["track_id"] == tid
    )
    votes = class_votes[tid]
    total_votes = sum(votes.values())
    id_summary.append({
        "track_id": tid,
        "locked_class": locked_class_by_id[tid],
        "n_detections": count,
        "class_counts": dict(votes),
        "class_purity": round(votes.get(locked_class_by_id[tid], 0) / total_votes, 3) if total_votes else None,
        "first_frame": frames_present[0],
        "last_frame": frames_present[-1],
        "frame_span": frames_present[-1] - frames_present[0] + 1,
        "coverage_ratio": round(count / (frames_present[-1] - frames_present[0] + 1), 3),
    })

id_summary.sort(key=lambda r: (r["locked_class"], -r["frame_span"]))

print(f"{'id':>5} {'class':<10} {'n_det':>7} {'span':>8} {'cover':>6}  class_counts")
for r in id_summary:
    print(f"{r['track_id']:>5} {r['locked_class']:<10} {r['n_detections']:>7} "
          f"{r['frame_span']:>8} {r['coverage_ratio']:>6.2f}  {r['class_counts']}")

print(f"\ntotal IDs: {len(id_summary)}")
for cls in set(r["locked_class"] for r in id_summary):
    cls_rows = [r for r in id_summary if r["locked_class"] == cls]
    print(f"  {cls}: {len(cls_rows)} IDs, "
          f"median coverage_ratio={sorted(r['coverage_ratio'] for r in cls_rows)[len(cls_rows)//2]:.2f}")

# low coverage_ratio (detections spread thin over a huge frame span) is a cheap proxy for
# "this ID is probably several stitched-together tracklets" -- worth auditing first
low_coverage = sorted([r for r in id_summary if r["coverage_ratio"] < 0.05], key=lambda r: r["coverage_ratio"])
print(f"\nIDs with coverage_ratio < 0.05 (prioritize these in the audit notebook): {len(low_coverage)}")
for r in low_coverage[:20]:
    print(f"  id={r['track_id']} class={r['locked_class']} n_det={r['n_detections']} span={r['frame_span']} cover={r['coverage_ratio']}")


   id class        n_det     span  cover  class_counts
   26 goalkeeper    2278    69605   0.03  {'goalkeeper': 2278}
   25 goalkeeper    1150    66969   0.02  {'goalkeeper': 1150}
   28 goalkeeper   13553    64010   0.21  {'goalkeeper': 13549, 'referee': 2, 'player': 2}
   47 goalkeeper    1641    58683   0.03  {'goalkeeper': 1639, 'referee': 1, 'player': 1}
   52 goalkeeper    6973    57440   0.12  {'goalkeeper': 6970, 'player': 3}
   43 goalkeeper     672    57377   0.01  {'goalkeeper': 672}
   37 goalkeeper    1395    54434   0.03  {'goalkeeper': 1395}
   40 goalkeeper    2848    54345   0.05  {'goalkeeper': 2847, 'player': 1}
   39 goalkeeper    1138    53143   0.02  {'goalkeeper': 1137, 'player': 1}
   45 goalkeeper     802    50630   0.02  {'goalkeeper': 801, 'player': 1}
   27 goalkeeper    2145    49306   0.04  {'player': 2, 'goalkeeper': 2143}
   58 goalkeeper    6977    49287   0.14  {'goalkeeper': 6976, 'player': 1}
   46 goalkeeper    1160    47137   0.03  {'goalkeeper': 1

## Cell 13b -- Verify split: id-count table, referee-locked tracks vs fresh ft3 class

Same diagnostic that originally found track 2 and track 41's contamination -- run again here, AFTER Cell 12b's split, as verification. Every referee-locked track should now show player_count and goalkeeper_count at or near 0 (a few stray frames from ordinary detector noise are fine; a large count means the split frame was wrong or missed a track).

In [15]:
# ---- Cell 13b (verification) -- Purely diagnostic, does NOT modify
# final_tracking_cache or locked_class_by_id. In-memory only (IoU lookups against
# detection_cache, already loaded) -- no video decode, no live model call.

IOU_MATCH_MIN = 0.5

def best_iou_class(frame_idx, bbox, detection_cache, iou_min=IOU_MATCH_MIN):
    best_iou, best_class = 0.0, None
    for d in detection_cache.get(frame_idx, []):
        if d["class"] not in CLASS_TO_ID:
            continue
        score = iou(bbox, d["bbox"])
        if score > best_iou:
            best_iou, best_class = score, d["class"]
    return best_class if best_iou >= iou_min else None


referee_ids = [tid for tid, cls in locked_class_by_id.items() if cls == "referee"]

rows = []
for tid in referee_ids:
    frames = sorted(
        (f, t["bbox"]) for f, data in final_tracking_cache.items()
        for t in data["tracks"] if t["track_id"] == tid
    )
    n_player, n_referee, n_goalkeeper, n_no_match = 0, 0, 0, 0
    for f, bbox in frames:
        cls = best_iou_class(f, bbox, detection_cache)
        if cls == "player":
            n_player += 1
        elif cls == "referee":
            n_referee += 1
        elif cls == "goalkeeper":
            n_goalkeeper += 1
        else:
            n_no_match += 1

    rows.append({
        "id": tid, "n_frames": len(frames),
        "player_count": n_player, "player_frac": round(n_player / len(frames), 3) if frames else 0,
        "goalkeeper_count": n_goalkeeper, "referee_count": n_referee, "no_match": n_no_match,
    })

rows.sort(key=lambda r: -r["player_count"])

print(f"{'id':>5} {'n_frames':>9} {'player_count':>13} {'player_frac':>12} "
      f"{'goalkeeper_count':>16} {'referee_count':>14} {'no_match':>9}")
for r in rows:
    print(f"{r['id']:>5} {r['n_frames']:>9} {r['player_count']:>13} {r['player_frac']:>12} "
          f"{r['goalkeeper_count']:>16} {r['referee_count']:>14} {r['no_match']:>9}")

   id  n_frames  player_count  player_frac goalkeeper_count  referee_count  no_match
   60     22621           181        0.008                0          22440         0
   87     11175            21        0.002                0          11154         0
    2     30400             6          0.0                0          30394         0
   55      6662             5        0.001                0           6657         0
   41      8327             3          0.0                0           8324         0
    1     20736             1          0.0                0          20735         0
    3     12569             0          0.0                0          12569         0
   29      5667             0          0.0                0           5667         0
   31      7979             0          0.0                0           7979         0
   35      2033             0          0.0                0           2033         0
   36      3453             0          0.0                0      

## Cell 14 -- Save final cache

In [16]:
with open(OUTPUT_TRACKING_CACHE_PATH, "wb") as f:
    pickle.dump({"tracking_cache": final_tracking_cache, "locked_class_by_id": locked_class_by_id}, f)

print(f"Saved to {OUTPUT_TRACKING_CACHE_PATH}")
print(f"File size: {os.path.getsize(OUTPUT_TRACKING_CACHE_PATH) / (1024*1024):.1f} MB")


Saved to /kaggle/working/barca_atletico_tracking_cache_final.pkl
File size: 103.6 MB
